In [2]:
from pathlib import Path
from itertools import product
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm, chi2
from arch.bootstrap import optimal_block_length

input_file = Path("SP500_daily_forecasts.csv")
factor_file = Path("SP500_factors.csv")
output_directory = Path("dependence_robust_inference")
tail_probabilities = [0.01, 0.03, 0.05]
forecast_horizons = [1, 5, 10, 21]
sensitivity_bandwidths = [5, 10, 21, 63, 126, 252, 504]
bootstrap_replications = 2000
seed = 20261025
significance_level = 0.05
factor_date_column = "VaR_Estimation_Date"
samples = {
    "Full sample": (None, None),
    "Global Financial Crisis": ("2008-01-01", "2009-12-31"),
    "COVID-19": ("2020-01-01", "2020-12-31"),
    "Inflation": ("2022-01-01", "2022-12-31"),
}
original_bandwidths = {
    "Full sample": 10,
    "Global Financial Crisis": 5,
    "COVID-19": 5,
    "Inflation": 5,
}
term_labels = {
    "const": "Intercept",
    "VaR_Ratio": "VER",
    "log_sigma": "Log conditional volatility",
    "log_rv_iv": "Log realised-to-implied volatility ratio",
    "Breach_FHS": "Breach–non-breach mean VER difference",
    "IV": "Implied volatility",
    "VoV": "Volatility of volatility",
    "Credit_Spread": "Credit-spread proxy",
    "Yield_Curve": "Yield-curve proxy",
    "Realized_Vol": "Realised volatility",
}
previous_panel_results = {
    name: globals().get(name)
    for name in ["ols_nw"]
}
existing_factors = globals().get("factors")
existing_rv_controls = globals().get("rv_iv_factors")
coefficient_records = []
joint_records = []
setting_records = []
inventory_records = []


def load_forecasts():
    data = pd.read_csv(input_file)
    required = [
        "RollingEnd",
        "VaR_Estimation_Date",
        "Alpha",
        "ForecastHorizon",
        "VaR_Ratio",
        "Sigma",
        "Breach_FHS",
    ]
    missing = sorted(set(required) - set(data.columns))

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for column in ["RollingEnd", "VaR_Estimation_Date"]:
        data[column] = pd.to_datetime(
            data[column], errors="raise"
        ).dt.tz_localize(None).dt.normalize()

    for column in required[2:]:
        data[column] = pd.to_numeric(data[column], errors="raise")

    if data[required].isna().any().any():
        raise ValueError("Required forecast fields contain missing values.")

    if not np.isfinite(
        data[required[2:]].to_numpy(dtype=float)
    ).all():
        raise ValueError("Nonfinite forecast values.")

    if not data["Breach_FHS"].isin([0, 1]).all():
        raise ValueError("Invalid breach indicators.")

    if not data["Sigma"].gt(0).all():
        raise ValueError("Conditional volatility must be positive.")

    if data.duplicated(
        ["RollingEnd", "Alpha", "ForecastHorizon"]
    ).any():
        raise ValueError("Duplicate forecast observations.")

    data = data.loc[
        data["ForecastHorizon"].isin(forecast_horizons)
        & np.isclose(
            data["Alpha"].to_numpy()[:, None],
            np.asarray(tail_probabilities)[None, :],
        ).any(axis=1)
    ].copy()

    data["log_sigma"] = np.log(data["Sigma"])
    return data.sort_values(
        ["VaR_Estimation_Date", "Alpha", "ForecastHorizon"]
    ).reset_index(drop=True)


def prepare_factor_frame(frame):
    frame = frame.copy()

    if not isinstance(frame.index, pd.DatetimeIndex):
        candidates = [
            column for column in frame.columns
            if str(column).lower() in [
                "date", "forecast origin", "evaluation date",
                "var_estimation_date",
            ]
        ]

        if not candidates:
            raise ValueError("Factor data require a date index or date column.")

        frame = frame.set_index(candidates[0])

    frame.index = pd.DatetimeIndex(
        pd.to_datetime(frame.index)
    ).tz_localize(None).normalize()

    if frame.index.has_duplicates:
        raise ValueError("Duplicate factor dates.")

    return frame.sort_index()


def add_factors(data):
    frames = []

    if isinstance(existing_factors, pd.DataFrame):
        frames.append(prepare_factor_frame(existing_factors))

    if isinstance(existing_rv_controls, pd.DataFrame):
        frames.append(prepare_factor_frame(existing_rv_controls))

    if factor_file.exists():
        frames.append(prepare_factor_frame(
            pd.read_csv(factor_file, index_col=0, parse_dates=True)
        ))

    if not frames:
        return data, False

    combined = frames[0]

    for frame in frames[1:]:
        for column in frame.columns:
            if column not in combined.columns:
                combined = combined.join(frame[[column]], how="outer")

    needed = [
        column for column in [
            "IV", "VoV", "Credit_Spread", "Yield_Curve",
            "Realized_Vol", "Log_RV_IV_Ratio",
        ]
        if column in combined.columns and column not in data.columns
    ]

    if needed:
        data = data.merge(
            combined[needed],
            left_on=factor_date_column,
            right_index=True,
            how="left",
            validate="many_to_one",
        )

    if "Log_RV_IV_Ratio" in data.columns:
        data["log_rv_iv"] = pd.to_numeric(
            data["Log_RV_IV_Ratio"], errors="coerce"
        )
    elif {"IV", "Realized_Vol"}.issubset(data.columns):
        valid = data["IV"].gt(0) & data["Realized_Vol"].gt(0)
        data["log_rv_iv"] = np.nan
        data.loc[valid, "log_rv_iv"] = np.log(
            data.loc[valid, "Realized_Vol"]
            / data.loc[valid, "IV"]
        )

    return data, True


def select_sample(data, sample):
    start, end = samples[sample]

    if start is None:
        return data.copy()

    return data.loc[
        data["VaR_Estimation_Date"].between(start, end)
    ].copy()


def aggregate_scores(scores, dates, date_grid):
    frame = pd.DataFrame(
        np.asarray(scores, dtype=float),
        index=pd.DatetimeIndex(dates),
    )
    return (
        frame.groupby(level=0).sum()
        .reindex(date_grid, fill_value=0)
        .to_numpy(dtype=float)
    )


def select_bandwidth(score_series):
    centered = score_series - score_series.mean(axis=0)
    recommendations = []

    for column in centered.T:
        if len(column) < 10 or np.dot(column[:-1], column[:-1]) <= 1e-20:
            continue

        rho = float(
            np.dot(column[:-1], column[1:])
            / np.dot(column[:-1], column[:-1])
        )
        rho = np.clip(rho, -0.97, 0.97)
        persistence = 4 * rho ** 2 / (1 - rho ** 2) ** 2
        recommendation = 1.1447 * (
            persistence * len(column)
        ) ** (1 / 3)

        if np.isfinite(recommendation):
            recommendations.append(recommendation)

    automatic = max(1, int(np.ceil(
        max(recommendations, default=1)
    )))
    cap = max(1, len(score_series) // 4)
    primary = min(automatic, cap)
    return automatic, primary, automatic > cap


def hac_covariance(score_series, bread, bandwidth):
    centered = score_series - score_series.mean(axis=0)
    meat = centered.T @ centered
    bandwidth = min(int(bandwidth), len(centered) - 1)

    for lag in range(1, bandwidth + 1):
        weight = 1 - lag / (bandwidth + 1)
        cross = centered[lag:].T @ centered[:-lag]
        meat += weight * (cross + cross.T)

    covariance = bread @ meat @ bread.T
    return (covariance + covariance.T) / 2


def fit_and_report(
    data,
    response,
    predictors,
    family,
    analysis,
    sample,
    date_grid,
    alpha=np.nan,
    horizon=np.nan,
    original_lags=None,
    joint_groups=None,
):
    required = [response, "VaR_Estimation_Date"] + predictors
    available = data[required].replace(
        [np.inf, -np.inf], np.nan
    ).dropna().sort_values("VaR_Estimation_Date")

    base = {
        "Analysis": analysis,
        "Model": family,
        "Sample": sample,
        "Tail probability": alpha,
        "Forecast horizon (trading days)": horizon,
    }

    if len(available) < len(predictors) + 10:
        inventory_records.append({
            **base,
            "Status": "Not estimated: insufficient observations",
        })
        return

    if available[response].nunique() < 2:
        inventory_records.append({
            **base,
            "Status": "Not estimated: no outcome variation",
        })
        return

    design = sm.add_constant(
        available[predictors].astype(float),
        has_constant="add",
    )
    outcome = available[response].astype(float)

    if np.linalg.matrix_rank(design) < design.shape[1]:
        inventory_records.append({
            **base,
            "Status": "Not estimated: rank-deficient design",
        })
        return

    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            if family == "OLS":
                result = sm.OLS(outcome, design).fit()
                scores = design.to_numpy() * (
                    result.resid.to_numpy()[:, None]
                )
                bread = np.linalg.inv(
                    design.to_numpy().T @ design.to_numpy()
                )
            else:
                constructor = sm.Logit if family == "Logit" else sm.Probit
                result = constructor(outcome, design).fit(
                    disp=False, maxiter=1000
                )

                if not result.mle_retvals.get("converged", False):
                    raise ValueError("Optimizer did not converge.")

                if any(
                    "separation" in str(item.message).lower()
                    for item in caught
                ):
                    raise ValueError("Separation detected.")

                scores = result.model.score_obs(result.params)
                bread = np.linalg.inv(
                    -result.model.hessian(result.params)
                )

        if not np.isfinite(
            np.r_[result.params, bread.ravel(), scores.ravel()]
        ).all():
            raise ValueError("Nonfinite estimates or scores.")

    except Exception as exc:
        inventory_records.append({
            **base,
            "Status": f"Not estimated: {type(exc).__name__}: {exc}",
        })
        return

    daily_scores = aggregate_scores(
        scores,
        available["VaR_Estimation_Date"],
        date_grid,
    )
    automatic, primary, capped = select_bandwidth(daily_scores)
    original_lags = (
        original_bandwidths[sample]
        if original_lags is None else int(original_lags)
    )

    bandwidths = sorted(set(
        [primary, original_lags]
        + [
            lag for lag in sensitivity_bandwidths
            if lag <= max(1, len(date_grid) // 4)
        ]
    ))
    bandwidths = [
        lag for lag in bandwidths
        if 0 <= lag < len(date_grid)
    ]

    setting_records.append({
        **base,
        "Observations": len(available),
        "Trading dates": len(date_grid),
        "Automatic bandwidth": automatic,
        "Primary bandwidth": primary,
        "Original bandwidth": original_lags,
        "Sensitivity bandwidths": ",".join(map(str, bandwidths)),
        "Bandwidth capped": capped,
        "Status": (
            "Short-series uncertainty; inspect sensitivity"
            if len(date_grid) < 500
            else "Inspect bandwidth sensitivity"
        ),
    })

    for bandwidth in bandwidths:
        covariance = hac_covariance(
            daily_scores, bread, bandwidth
        )
        diagonal = np.diag(covariance)
        standard_errors = np.sqrt(
            np.where(diagonal >= 0, diagonal, np.nan)
        )

        for index, term in enumerate(design.columns):
            estimate = float(result.params.iloc[index])
            standard_error = float(standard_errors[index])

            if np.isfinite(standard_error) and standard_error > 0:
                statistic = estimate / standard_error
                p_value = float(2 * norm.sf(abs(statistic)))
                lower = estimate - norm.ppf(0.975) * standard_error
                upper = estimate + norm.ppf(0.975) * standard_error
            else:
                p_value = lower = upper = np.nan

            coefficient_records.append({
                **base,
                "Term": term_labels.get(term, term),
                "Observations": len(available),
                "Trading dates": len(date_grid),
                "Estimate": estimate,
                "HAC bandwidth": bandwidth,
                "Primary specification": bandwidth == primary,
                "Original specification": bandwidth == original_lags,
                "Standard error": standard_error,
                "95% lower": lower,
                "95% upper": upper,
                "p-value": p_value,
                "Bandwidth/date ratio": bandwidth / len(date_grid),
            })

        if joint_groups:
            for label, terms in joint_groups.items():
                indices = [
                    design.columns.get_loc(term)
                    for term in terms if term in design.columns
                ]

                if not indices:
                    continue

                coefficients = result.params.to_numpy()[indices]
                joint_covariance = covariance[np.ix_(indices, indices)]
                rank = np.linalg.matrix_rank(joint_covariance)

                if rank == len(indices):
                    statistic = float(
                        coefficients
                        @ np.linalg.solve(joint_covariance, coefficients)
                    )
                    p_value = float(
                        chi2.sf(statistic, len(indices))
                    )
                else:
                    statistic = p_value = np.nan

                joint_records.append({
                    **base,
                    "Hypothesis": label,
                    "HAC bandwidth": bandwidth,
                    "Primary specification": bandwidth == primary,
                    "Wald statistic": statistic,
                    "Degrees of freedom": len(indices),
                    "p-value": p_value,
                })

    inventory_records.append({
        **base,
        "Status": "Estimated with date-aggregated Bartlett HAC",
    })


def run_regressions(data):
    multifactor_columns = [
        "VaR_Ratio", "IV", "VoV",
        "Credit_Spread", "Yield_Curve", "Realized_Vol",
    ]

    for sample in samples:
        scoped = select_sample(data, sample)
        date_grid = pd.DatetimeIndex(
            scoped["VaR_Estimation_Date"].unique()
        ).sort_values()

        if len(date_grid) < 10:
            continue

        panel = scoped.copy()
        panel_predictors = []

        for alpha in [0.03, 0.01]:
            name = f"Tail probability {alpha:.0%}"
            panel[name] = np.isclose(panel["Alpha"], alpha).astype(float)
            panel_predictors.append(name)

        for horizon in [5, 10, 21]:
            name = f"Forecast horizon {horizon}"
            panel[name] = panel["ForecastHorizon"].eq(horizon).astype(float)
            panel_predictors.append(name)

        fit_and_report(
            panel,
            "VaR_Ratio",
            panel_predictors,
            "OLS",
            "VER panel regression",
            sample,
            date_grid,
            original_lags=5,
            joint_groups={
                "Joint tail-probability effects": panel_predictors[:2],
                "Joint forecast-horizon effects": panel_predictors[2:],
            },
        )

        for alpha, horizon in product(
            tail_probabilities, forecast_horizons
        ):
            subset = scoped.loc[
                np.isclose(scoped["Alpha"], alpha)
                & scoped["ForecastHorizon"].eq(horizon)
            ].copy()

            fit_and_report(
                subset,
                "VaR_Ratio",
                ["Breach_FHS"],
                "OLS",
                "Breach–non-breach VER difference",
                sample,
                date_grid,
                alpha,
                horizon,
            )

            specifications = {
                "VER-only breach regression": ["VaR_Ratio"],
                "Conditional-volatility control": [
                    "VaR_Ratio", "log_sigma"
                ],
                "Realised-to-implied volatility control": [
                    "VaR_Ratio", "log_sigma", "log_rv_iv"
                ],
            }

            for analysis, predictors in specifications.items():
                missing = [
                    name for name in predictors
                    if name not in subset.columns
                ]

                if missing:
                    inventory_records.append({
                        "Analysis": analysis,
                        "Sample": sample,
                        "Tail probability": alpha,
                        "Forecast horizon (trading days)": horizon,
                        "Status": f"Unavailable controls: {missing}",
                    })
                    continue

                for family in ["Logit", "Probit"]:
                    fit_and_report(
                        subset,
                        "Breach_FHS",
                        predictors,
                        family,
                        analysis,
                        sample,
                        date_grid,
                        alpha,
                        horizon,
                    )

            if np.isclose(alpha, 0.03) and horizon == 1:
                missing = [
                    name for name in multifactor_columns
                    if name not in subset.columns
                ]

                if missing:
                    inventory_records.append({
                        "Analysis": "Multifactor breach regression",
                        "Sample": sample,
                        "Tail probability": alpha,
                        "Forecast horizon (trading days)": horizon,
                        "Status": f"Unavailable controls: {missing}",
                    })
                else:
                    fit_and_report(
                        subset,
                        "Breach_FHS",
                        multifactor_columns,
                        "Logit",
                        "Multifactor breach regression",
                        sample,
                        date_grid,
                        alpha,
                        horizon,
                        original_lags=1,
                    )

        print(f"{sample}: regression inference completed", flush=True)


def circular_indices(size, length, generator):
    starts = generator.integers(
        0, size, size=int(np.ceil(size / length))
    )
    return (
        (starts[:, None] + np.arange(length)) % size
    ).ravel()[:size]


def coverage_checks(data):
    records = []

    for sample_index, sample in enumerate(samples):
        scoped = select_sample(data, sample)

        for alpha_index, alpha in enumerate(tail_probabilities):
            for horizon in forecast_horizons:
                subset = scoped.loc[
                    np.isclose(scoped["Alpha"], alpha)
                    & scoped["ForecastHorizon"].eq(horizon)
                ].sort_values("VaR_Estimation_Date")

                for breach_column, benchmark_name in [
                    ("Breach_FHS", "Model VaR"),
                    ("Breach_Cantelli", "Cantelli benchmark"),
                ]:
                    if breach_column not in subset.columns:
                        continue

                    outcomes = pd.to_numeric(
                        subset[breach_column], errors="coerce"
                    ).dropna().to_numpy(dtype=float)

                    base = {
                        "Sample": sample,
                        "Tail probability": alpha,
                        "Forecast horizon (trading days)": horizon,
                        "Benchmark": benchmark_name,
                        "Observations": len(outcomes),
                        "Breaches": int(outcomes.sum()),
                    }

                    if len(outcomes) < 25 or np.unique(outcomes).size < 2:
                        records.append({
                            **base,
                            "Observed breach frequency": (
                                outcomes.mean() if len(outcomes) else np.nan
                            ),
                            "Status": (
                                "Descriptive only: insufficient variation "
                                "for block-bootstrap coverage inference"
                            ),
                        })
                        continue

                    automatic = optimal_block_length(
                        outcomes[:, None]
                    )["circular"].to_numpy(dtype=float)
                    automatic = automatic[
                        np.isfinite(automatic) & (automatic > 0)
                    ]

                    if not len(automatic):
                        records.append({
                            **base,
                            "Observed breach frequency": outcomes.mean(),
                            "Status": "Automatic block selection unavailable",
                        })
                        continue

                    recommendation = max(
                        1, int(np.ceil(automatic.max()))
                    )
                    cap = max(1, len(outcomes) // 4)
                    primary = min(max(5, recommendation), cap)
                    lengths = sorted(set([
                        max(1, primary // 2),
                        primary,
                        min(cap, primary * 2),
                    ]))

                    for length in lengths:
                        generator = np.random.default_rng(
                            np.random.SeedSequence([
                                seed, sample_index, alpha_index,
                                horizon, length,
                                int(breach_column == "Breach_Cantelli"),
                            ])
                        )
                        means = np.empty(bootstrap_replications)

                        for replication in range(bootstrap_replications):
                            indices = circular_indices(
                                len(outcomes), length, generator
                            )
                            means[replication] = outcomes[indices].mean()

                        observed = float(outcomes.mean())
                        deviation = abs(observed - alpha)
                        centered_deviations = np.abs(means - observed)
                        p_value = (
                            1 + np.sum(centered_deviations >= deviation)
                        ) / (bootstrap_replications + 1)

                        records.append({
                            **base,
                            "Observed breach frequency": observed,
                            "Target breach frequency": alpha,
                            "Coverage difference": observed - alpha,
                            "Block length": length,
                            "Primary specification": length == primary,
                            "95% frequency lower": np.quantile(means, 0.025),
                            "95% frequency upper": np.quantile(means, 0.975),
                            "Coverage p-value": p_value,
                            "Automatic block recommendation": recommendation,
                            "Block recommendation capped": recommendation > cap,
                            "Status": (
                                "Approximate centered block-bootstrap "
                                "test of marginal coverage"
                            ),
                        })

        print(f"{sample}: coverage checks completed", flush=True)

    return pd.DataFrame(records)


def make_compact_summary(details):
    if details.empty:
        return pd.DataFrame()

    selected = details.loc[
        details["Term"].isin([
            "VER",
            "Breach–non-breach mean VER difference",
        ])
    ].copy()

    keys = [
        "Analysis", "Model", "Sample",
        "Tail probability", "Forecast horizon (trading days)",
        "Term",
    ]
    records = []

    for key, group in selected.groupby(keys, dropna=False):
        primary = group.loc[group["Primary specification"]]

        if primary.empty:
            continue

        row = dict(zip(keys, key))
        first = primary.iloc[0]
        valid = group["p-value"].dropna()

        row.update({
            "Observations": first["Observations"],
            "Estimate": first["Estimate"],
            "Primary bandwidth": first["HAC bandwidth"],
            "Primary standard error": first["Standard error"],
            "Primary 95% lower": first["95% lower"],
            "Primary 95% upper": first["95% upper"],
            "Primary p-value": first["p-value"],
            "Lowest p-value across bandwidths": (
                valid.min() if len(valid) else np.nan
            ),
            "Highest p-value across bandwidths": (
                valid.max() if len(valid) else np.nan
            ),
            "Significant at 5% across every reported bandwidth": (
                bool(valid.lt(significance_level).all())
                if len(valid) == len(group) else np.nan
            ),
        })

        for _, estimate in group.iterrows():
            lag = int(estimate["HAC bandwidth"])
            row[f"p-value: {lag} lags"] = estimate["p-value"]

        records.append(row)

    return pd.DataFrame(records)


def format_p_value(value):
    if pd.isna(value):
        return "N/A"
    return "< 0.001" if value < 0.001 else f"{value:.3f}"


def export_report(table, filename):
    report = table.copy()

    for column in report.columns:
        if "p-value" in column.lower():
            report[column] = report[column].map(format_p_value)

    report.to_csv(
        output_directory / filename,
        index=False,
        na_rep="N/A",
        float_format="%.8g",
    )


def main():
    output_directory.mkdir(parents=True, exist_ok=True)
    data = load_forecasts()
    data, factors_available = add_factors(data)

    run_regressions(data)
    coverage = coverage_checks(data)

    coefficients = pd.DataFrame(coefficient_records)
    joint_tests = pd.DataFrame(joint_records)
    settings = pd.DataFrame(setting_records)

    inventory_records.extend([
        {
            "Analysis": "Quartile breach frequencies",
            "Status": "Descriptive; no independence-based significance claim",
        },
        {
            "Analysis": "Markov transition probabilities",
            "Status": "Descriptive; rolling-window persistence acknowledged",
        },
        {
            "Analysis": "Ordinary KS and two-sample t-tests",
            "Status": (
                "Replace inferential claims about mean VER differences "
                "with the HAC mean-difference results; KS results descriptive"
            ),
        },
        {
            "Analysis": "Conventional Christoffersen independence p-values",
            "Status": (
                "Not recalibrated by this script; retain as descriptive "
                "diagnostics without dependence-robust significance claims"
            ),
        },
        {
            "Analysis": "Original Kupiec coverage p-values",
            "Status": (
                "Compare with the separate block-bootstrap marginal "
                "coverage results; do not relabel original p-values"
            ),
        },
        {
            "Analysis": "Rolling residual diagnostic rejection shares",
            "Status": (
                "Descriptive across overlapping windows; not independent tests"
            ),
        },
        {
            "Analysis": "Out-of-sample forecast uncertainty",
            "Status": (
                "Use the existing paired circular-block bootstrap results"
            ),
        },
        {
            "Analysis": "Factor availability",
            "Status": (
                "Factor data supplied; inspect model-level inventory"
                if factors_available
                else "Factor-dependent regressions not reproduced"
            ),
        },
    ])

    inventory = pd.DataFrame(inventory_records)
    compact = make_compact_summary(coefficients)

    coefficients.to_csv(
        output_directory / "Regression_inference_numeric.csv",
        index=False,
        na_rep="N/A",
    )
    coverage.to_csv(
        output_directory / "Coverage_inference_numeric.csv",
        index=False,
        na_rep="N/A",
    )

    export_report(
        compact, "Inference_sensitivity_summary.csv"
    )
    export_report(
        coefficients, "Regression_inference_report.csv"
    )
    export_report(
        joint_tests, "Panel_joint_tests.csv"
    )
    export_report(
        coverage, "Coverage_block_bootstrap.csv"
    )
    export_report(
        settings, "Inference_settings.csv"
    )

    inventory.to_csv(
        output_directory / "Reporting_inventory.csv",
        index=False,
        na_rep="N/A",
    )


    print(f"Results saved to {output_directory.resolve()}")
   
    


if __name__ == "__main__":
    main()

Full sample: regression inference completed
Global Financial Crisis: regression inference completed
COVID-19: regression inference completed
Inflation: regression inference completed
Full sample: coverage checks completed
Global Financial Crisis: coverage checks completed
COVID-19: coverage checks completed
Inflation: coverage checks completed
Results saved to C:\Users\PC\Documents\VER\code_2nd_submission\dependence_robust_inference
